&nbsp;

# PRJDB12380 and PRJDB9451 process

- [Remarkable improvement in detection of readthrough downstream-of-gene transcripts by semi-extractable RNA-sequencing](https://doi.org/10.1261/rna.079469.122)
- Book (Japanese): [“情報”から“マテリアル”へ　ノンコーディングRNA研究](https://www.yodosha.co.jp/yodobook/book/9784758104210/)
- `STAR` によるマッピングと、`featurecounts` によるリードカウント結果を DESeq2 に流す

In [1]:
import re
import polars as pl
import pandas as pd
import anndata as ad
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)

polars.config.Config

&nbsp;

&nbsp;

&nbsp;

## PRJDB12380 Metadata

- [DDBJ Search: PRJDB12380](https://ddbj.nig.ac.jp/search/entry/bioproject/PRJDB12380/)

In [2]:
paired_metadata_df_prjdb12380 = pl.read_csv(
    "../../Data/metadata/metadata_paired/PRJDB12380_paired_metadata.tsv",
    separator="\t"
)

display(paired_metadata_df_prjdb12380)

sample_heat_id,sample_control_id,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line,library_name,tag,first_created,experiment_accession,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR321028""","""DRR321026""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407109 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407107""","""SAMD00407109 vs SAMD00407107""","""Homo sapiens""","""DRA012807""","""PolyA""","""2022-12-03""","""593541539 vs 695433398""","""998331814 vs 1204959675""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""Heat stressed and recovered WT HAP1 RNA prepared by conventional method vs WT HAP1 RNA prepared by conventional method""","""DRP009294""","""48176699 vs 60317874""","""Searching for stress inducible architectural noncoding RNAs""","""GENERATED_FILE""","""Heat stressed and recovered WT HAP1 RNA prepared by conventional method vs WT HAP1 RNA prepared by conventional method""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407109 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407107""","""1734361164 vs 2171443464""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00407109 vs SAMD00407107""","""DRR321028 vs DRR321026""","""Truseq stranded mRNA Library Prep Kit(Illumina Inc.)""","""DRX310375 vs DRX310373""","""HOKUDAI""","""HAP1""","""WTHAP_HSrec_Conv vs WTHAP_Ctrl_Conv""","""env_tax;env_tax:terrestrial""","""2022-12-02""","""DRX310375 vs DRX310373""","""Homo sapiens""",9606,"""PRJDB12380""","""DRR321028 vs DRR321026""","""DRS266840 vs DRS266838""","""2022-12-02""","""DRP009294""","""TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)""","""https://doi.org/10.1261/rna.079469.122""","""42 C""","""120 min.""","""37 C""","""total RNA""","""WT"""
"""DRR321029""","""DRR321027""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407110 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407108""","""SAMD00407110 vs SAMD00407108""","""Homo sapiens""","""DRA012807""","""PolyA""","""2022-12-03""","""635880364 vs 963117646""","""1079612575 vs 1691264824""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""Heat stressed and recovered WT HAP1 RNA prepared by improved method vs WT HAP1 RNA prepared by improved method""","""DRP009294""","""51291309 vs 83087719""","""Searching for stress inducible architectural noncoding RNAs""","""GENERATED_FILE""","""Heat stressed and recovered WT HAP1 RNA prepared by improved method vs WT HAP1 RNA prepared by improved method""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407110 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00407108""","""1846487124 vs 2991157884""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00407110 vs SAMD00407108""","""DRR321029 vs DRR321027""","""Truseq stranded mRNA Library Prep Kit(Illumina Inc.)""","""DRX310376 vs DRX310374""","""HOKUDAI""","""HAP1""","""WTHAP_HSrec_Impr vs WTHAP_Ctrl_Impr""","""env_tax;env_tax:terrestrial""","""2022-12-02""","""DRX310

&nbsp;

&nbsp;

&nbsp;

## PRJDB9451 Metadata

- [DDBJ Search: PRJDB9451](https://ddbj.nig.ac.jp/search/entry/bioproject/PRJDB9451/)

In [3]:
paired_metadata_df_prjdb9451 = pl.read_csv(
    "../../Data/metadata/metadata_paired/PRJDB9451_paired_metadata.tsv",
    separator="\t"
)

display(paired_metadata_df_prjdb9451)

sample_heat_id,sample_control_id,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line,library_name,tag,first_created,experiment_accession,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,sample_molecule_ch1,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR214953""","""DRR214951""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209816 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209814""","""SAMD00209816 vs SAMD00209814""","""Homo sapiens""","""DRA009793""","""Inverse rRNA""","""2022-12-21 vs 2022-03-11""","""1468198927 vs 1861273855""","""2368491635 vs 3029873213""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""HAP1 cells culture at 42 degree for 2 hours and then at 37 degree for 1 hour, Total RNAs were extracted by conventional TRIregeant protocol vs HAP1 cells Normal culture condition, Total RNAs were extr…","""DRP008266""","""79583343 vs 100955243""","""Total RNA-seq of stressed cells upon conventional or improved RNA extraction""","""GENERATED_FILE""","""7_2_WTHAP_HSrec_Conv vs 1_2_WTHAP_Ctrl_Conv""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209816 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209814""","""3979167150 vs 5047762150""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00209816 vs SAMD00209814""","""DRR214953 vs DRR214951""","""TruSeq Stranded mRNA HT Sample Prep Kit (skip the polyA selection step)""","""DRX205288 vs DRX205286""","""HOKUDAI""","""HAP1 cell""","""7_2_WTHAP_HSrec_Conv vs 1_2_WTHAP_Ctrl_Conv""","""env_tax;env_tax:terrestrial""","""2022-03-08""","""DRX205288 vs DRX205286""","""Homo sapiens""",9606,"""PRJDB9451""","""DRR214953 vs DRR214951""","""DRS230493 vs DRS230491""","""2022-03-08""","""DRP008266""","""TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)""","""https://doi.org/10.1261/rna.079469.122""","""42 C""","""120 min.""","""37 C""","""total RNA""","""WT"""
"""DRR214954""","""DRR214952""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209817 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209815""","""SAMD00209817 vs SAMD00209815""","""Homo sapiens""","""DRA009793""","""Inverse rRNA""","""2022-03-11""","""2006387263 vs 1199349638""","""3354762464 vs 1988136569""","""ILLUMINA""","""ARCHIVAL_FILE""","""Generic""","""HAP1 cells culture at 42 degree for 2 hours and then at 37 degree for 1 hour, Total RNAs were extracted by improved TRIregeant protocol vs HAP1 cells Normal culture condition, Total RNAs were extracte…","""DRP008266""","""109242105 vs 64933636""","""Total RNA-seq of stressed cells upon conventional or improved RNA extraction""","""GENERATED_FILE""","""8_2_WTHAP_HSrec_Impr vs 2_2_WTHAP_Ctrl_Impr""","""Illumina HiSeq 3000""","""Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209817 vs Illumina HiSeq 3000 sequencing: Illumina HiSeq 3000 sequencing of SAMD00209815""","""5462105250 vs 3246681800""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00209817 vs SAMD00209815""","""DRR214954 vs DRR214952""","""TruSeq Stranded mRNA HT Sample Prep Kit (skip the polyA selection step)""","""DRX205

&nbsp;

&nbsp;

## `featureCounts` Result File (Gene Level)

- GENCODE release 49

In [4]:
ANNOTATION_COLUMNS = ["Geneid", "Chr", "Start", "End", "Strand", "Length"]
def featurecounts_column_name(name: str) -> str:
    # Keep annotation columns; turn a staged BAM path into its DRR accession.
    if name in ANNOTATION_COLUMNS:
        return name
    match = re.search(r"(DRR\d+)", name)
    return match.group(1) if match else name

In [5]:
prjdb12380_featurecounts_result = pl.read_csv(
    "../../out/transcript_classification/arcRNA_candidate/arcRNA_featurecounts/PRJDB12380_featureCounts.txt",
    separator="\t",
    n_threads=16,
    comment_prefix="#"
).with_columns(
    pl.col("Chr").str.split(";").list.unique(maintain_order=True).list.join(";"),
    pl.col("Strand").str.split(";").list.unique(maintain_order=True).list.join(";"),
).rename(
    featurecounts_column_name
).drop(
    "Start",
    "End"
)

display(prjdb12380_featurecounts_result)

Geneid,Chr,Strand,Length,DRR321026,DRR321027,DRR321028,DRR321029
str,str,str,i64,i64,i64,i64,i64
"""ENSG00000290825.2""","""chr1""","""+""",2898,0,0,0,0
"""ENSG00000223972.6""","""chr1""","""+""",632,0,0,0,0
"""ENSG00000310526.1""","""chr1""","""-""",6775,21,43,8,10
"""ENSG00000227232.6""","""chr1""","""-""",1380,0,0,0,0
"""ENSG00000278267.1""","""chr1""","""-""",68,0,0,0,0
…,…,…,…,…,…,…,…
"""ENSG00000198695.2""","""chrM""","""-""",525,697,1475,484,211
"""ENSG00000210194.1""","""chrM""","""-""",69,0,1,1,0
"""ENSG00000198727.2""","""chrM""","""+""",1141,7861,5704,15954,10787


In [6]:
prjdb9451_featurecounts_result = pl.read_csv(
    "../../out/transcript_classification/arcRNA_candidate/arcRNA_featurecounts/PRJDB9451_featureCounts.txt",
    separator="\t",
    n_threads=16,
    comment_prefix="#"
).with_columns(
    pl.col("Chr").str.split(";").list.unique(maintain_order=True).list.join(";"),
    pl.col("Strand").str.split(";").list.unique(maintain_order=True).list.join(";"),
).rename(
    featurecounts_column_name
).drop(
    "Start",
    "End"
)

display(prjdb9451_featurecounts_result)

Geneid,Chr,Strand,Length,DRR214951,DRR214952,DRR214953,DRR214954
str,str,str,i64,i64,i64,i64,i64
"""ENSG00000290825.2""","""chr1""","""+""",2898,0,0,0,0
"""ENSG00000223972.6""","""chr1""","""+""",632,0,0,0,0
"""ENSG00000310526.1""","""chr1""","""-""",6775,52,34,10,20
"""ENSG00000227232.6""","""chr1""","""-""",1380,0,0,0,0
"""ENSG00000278267.1""","""chr1""","""-""",68,0,0,0,0
…,…,…,…,…,…,…,…
"""ENSG00000198695.2""","""chrM""","""-""",525,1319,411,789,1138
"""ENSG00000210194.1""","""chrM""","""-""",69,6,3,1,6
"""ENSG00000198727.2""","""chrM""","""+""",1141,13054,10557,9919,6971


In [7]:
semi_extractable_gene_counts = prjdb12380_featurecounts_result.join(
    prjdb9451_featurecounts_result,
    on=["Geneid", "Chr", "Strand", "Length"],
    how="inner"
)

display(semi_extractable_gene_counts)

Geneid,Chr,Strand,Length,DRR321026,DRR321027,DRR321028,DRR321029,DRR214951,DRR214952,DRR214953,DRR214954
str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""ENSG00000290825.2""","""chr1""","""+""",2898,0,0,0,0,0,0,0,0
"""ENSG00000223972.6""","""chr1""","""+""",632,0,0,0,0,0,0,0,0
"""ENSG00000310526.1""","""chr1""","""-""",6775,21,43,8,10,52,34,10,20
"""ENSG00000227232.6""","""chr1""","""-""",1380,0,0,0,0,0,0,0,0
"""ENSG00000278267.1""","""chr1""","""-""",68,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000198695.2""","""chrM""","""-""",525,697,1475,484,211,1319,411,789,1138
"""ENSG00000210194.1""","""chrM""","""-""",69,0,1,1,0,6,3,1,6
"""ENSG00000198727.2""","""chrM""","""+""",1141,7861,5704,15954,10787,13054,10557,9919,6971


In [8]:
# featureCounts raw counts: genes x samples -> samples x genes
counts = (
    semi_extractable_gene_counts
    .drop("Chr", "Strand", "Length")
    .to_pandas()
    .set_index("Geneid")
    .T
)

# pyDESeq2 requires non-negative integers
counts = counts.round().astype(int)

display(counts)

Geneid,ENSG00000290825.2,ENSG00000223972.6,ENSG00000310526.1,ENSG00000227232.6,ENSG00000278267.1,ENSG00000243485.6,ENSG00000284332.1,ENSG00000237613.3,ENSG00000308361.1,ENSG00000290826.2,...,ENSG00000198886.2,ENSG00000210176.1,ENSG00000210184.1,ENSG00000210191.1,ENSG00000198786.2,ENSG00000198695.2,ENSG00000210194.1,ENSG00000198727.2,ENSG00000210195.2,ENSG00000210196.2
DRR321026,0,0,21,0,0,0,0,0,0,0,...,9596,313,18,6,5042,697,0,7861,16,100
DRR321027,0,0,43,0,0,0,0,0,0,0,...,5603,330,17,5,3692,1475,1,5704,22,104
DRR321028,0,0,8,0,0,0,0,0,0,0,...,22255,311,23,3,10183,484,1,15954,26,49
DRR321029,0,0,10,0,0,0,0,0,0,0,...,14530,323,10,7,6971,211,0,10787,35,23
DRR214951,0,0,52,0,0,0,0,0,0,0,...,20614,215,152,7,10844,1319,6,13054,141,240
DRR214952,0,0,34,0,0,1,0,0,0,0,...,16880,30,51,4,9574,411,3,10557,30,62
DRR214953,0,0,10,0,0,0,0,0,0,0,...,15318,139,129,11,7051,789,1,9919,70,109
DRR214954,0,0,20,0,0,0,0,0,0,0,...,12199,128,92,7,5985,1138,6,6971,50,89


In [9]:
# Build sample metadata (index must match counts.index)
metadata = pd.DataFrame(index=counts.index)
metadata.index.name = "sample"

display(metadata)

""
sample
DRR321026
DRR321027
DRR321028
DRR321029
DRR214951
DRR214952
DRR214953
DRR214954


In [10]:
# Expand paired metadata into per-sample rows
rows = []
for df, project in [
    (paired_metadata_df_prjdb12380, "PRJDB12380"),
    (paired_metadata_df_prjdb9451, "PRJDB9451"),
]:
    for row in df.iter_rows(named=True):
        heat_library_name, control_library_name = row["library_name"].split(" vs ")

        def method_from_library_name(library_name: str) -> str:
            return "improved" if "_Impr" in library_name else "conventional"

        for sid, condition, library_name in [
            (row["sample_heat_id"], "heat", heat_library_name),
            (row["sample_control_id"], "control", control_library_name),
        ]:
            rows.append({
                "sample": sid,
                "project": project,
                "condition": condition,
                "method": method_from_library_name(library_name),
            })

metadata = pd.DataFrame(rows).set_index("sample").loc[counts.index]

display(metadata)

,project,condition,method
sample,,,
DRR321026,PRJDB12380,control,conventional
DRR321027,PRJDB12380,control,improved
DRR321028,PRJDB12380,heat,conventional
DRR321029,PRJDB12380,heat,improved
DRR214951,PRJDB9451,control,conventional
DRR214952,PRJDB9451,control,improved
DRR214953,PRJDB9451,heat,conventional
DRR214954,PRJDB9451,heat,improved


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Heat Condition Sample (improved vs. conventional)

In [11]:
# Heat samples only: compare improved vs conventional extraction
metadata_heat = metadata.query("condition == 'heat'")
counts_heat = counts.loc[metadata_heat.index]
genes_to_keep = counts_heat.columns[counts_heat.max(axis=0) > 10]
counts_heat = counts_heat[genes_to_keep]

adata_heat = ad.AnnData(
    X=counts_heat.values,
    obs=metadata_heat,
    var=pd.DataFrame(index=counts_heat.columns),
    uns={"counts_from_abundance": "no"},
)
adata_heat.obs_names = counts_heat.index.astype(str).tolist()
adata_heat.var_names = counts_heat.columns.astype(str).tolist()

display(metadata_heat)
display(adata_heat)

,project,condition,method
sample,,,
DRR321028,PRJDB12380,heat,conventional
DRR321029,PRJDB12380,heat,improved
DRR214953,PRJDB9451,heat,conventional
DRR214954,PRJDB9451,heat,improved


AnnData object with n_obs × n_vars = 4 × 24001
    obs: 'project', 'condition', 'method'
    uns: 'counts_from_abundance'
    layers: None (.X)

&nbsp;

&nbsp;

&nbsp;

## `pyDESeq2`: heat-condition sample (improved vs conventional)

- 対象: heat 処理後 4 サンプル（各プロジェクトで conventional / improved が 1 本ずつ）
- design: `~ project + method`（プロジェクト差を調整しつつ抽出法の効果を評価）
- contrast: `improved` vs `conventional`（正の log2FC = improved で高発現）

In [12]:
# https://pydeseq2.readthedocs.io/en/latest/auto_examples/plot_step_by_step.html
# /usr/local/lib/python3.13/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
# self.fit_dispersion_prior()

dds_heat = DeseqDataSet(
    adata=adata_heat,
    design="~ project + method", # プロジェクト差 (PRJDB12380 vs PRJDB9451) を調整しつつ, 抽出法 (improved vs conventional) の効果を評価
    refit_cooks=True,
    inference=DefaultInference(n_cpus=16),
    quiet=True,
    n_cpus=16
)

dds_heat.deseq2()

stat_result_heat = DeseqStats(
    dds_heat,
    contrast=["method", "improved", "conventional"],
    quiet=True,
)

stat_result_heat.summary()

results_heat_impr_vs_conv = stat_result_heat.results_df
results_heat_impr_vs_conv.index.name = "gene_id" # Add gene_id as index name

display(results_heat_impr_vs_conv.sort_values(by=["log2FoldChange", "padj"], ascending=(False, False)))

/usr/local/lib/python3.13/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_id,,,,,,
ENSG00000263511.1,14.462220,7.212557,2.503587,2.880889,0.003966,0.013993
ENSG00000303424.1,7.148988,6.190338,2.622069,2.360860,0.018233,NaN
ENSG00000308212.1,6.984743,6.143694,2.627547,2.338186,0.019378,NaN
ENSG00000259335.1,5.043049,5.734345,2.680789,2.139051,0.032431,NaN
ENSG00000278964.1,5.023762,5.691297,2.660681,2.139038,0.032433,NaN
...,...,...,...,...,...,...
ENSG00000307584.1,3.829557,-2.871111,1.867594,-1.537331,0.124212,NaN
ENSG00000233111.1,5.867954,-3.077619,1.735716,-1.773112,0.076210,NaN
ENSG00000298060.1,4.682098,-3.148305,1.661142,-1.895265,0.058057,NaN


In [13]:
results_heat_impr_vs_conv.to_csv(
    "../../out/transcript_classification/arcRNA_candidate/arcRNA_deseq2/pydeseq2_heat_impr_vs_conv_results.tsv", 
    sep="\t",
    index=True
)

In [14]:
results_heat_impr_vs_conv_impr = results_heat_impr_vs_conv[
    (results_heat_impr_vs_conv["padj"] < 0.001)
    & (results_heat_impr_vs_conv["log2FoldChange"] > 1)
]


display(results_heat_impr_vs_conv_impr)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_id,,,,,,
ENSG00000284747.1,207.826349,1.053892,0.252848,4.168091,3.071612e-05,2.834553e-04
ENSG00000224315.2,71.794088,2.147486,0.425731,5.044228,4.553557e-07,1.027558e-05
ENSG00000228423.2,61.063618,2.016640,0.450482,4.476630,7.583057e-06,9.268545e-05
ENSG00000226029.1,273.930236,1.353638,0.232860,5.813097,6.132741e-09,3.356574e-07
ENSG00000070831.19,1272.354478,1.295245,0.140951,9.189302,3.954001e-20,1.450752e-17
...,...,...,...,...,...,...
ENSG00000233874.2,138.355448,1.321611,0.291569,4.532750,5.822084e-06,7.539409e-05
ENSG00000241743.5,948.812959,1.551253,0.150567,10.302760,6.846802e-25,4.239241e-22
ENSG00000235244.7,910.055772,1.595317,0.156201,10.213204,1.730535e-24,1.008444e-21


In [15]:
# NEAT1
ENSG00000245532 = results_heat_impr_vs_conv.query("gene_id == 'ENSG00000245532.13'")


display(ENSG00000245532)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_id,,,,,,
ENSG00000245532.13,39301.154293,3.879836,0.114143,33.991087,3.017037e-253,5.977656e-249


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. Control condition (Improved vs. Conventional)

In [16]:
# Control samples only: compare improved vs conventional extraction
metadata_control = metadata.query("condition == 'control'")
counts_control = counts.loc[metadata_control.index]
genes_to_keep = counts_control.columns[counts_control.max(axis=0) > 10]
counts_control = counts_control[genes_to_keep]

adata_control = ad.AnnData(
    X=counts_control.values,
    obs=metadata_control,
    var=pd.DataFrame(index=counts_control.columns),
    uns={"counts_from_abundance": "no"},
)
adata_control.obs_names = counts_control.index.astype(str).tolist()
adata_control.var_names = counts_control.columns.astype(str).tolist()

display(metadata_control)
display(adata_control)

,project,condition,method
sample,,,
DRR321026,PRJDB12380,control,conventional
DRR321027,PRJDB12380,control,improved
DRR214951,PRJDB9451,control,conventional
DRR214952,PRJDB9451,control,improved


AnnData object with n_obs × n_vars = 4 × 22441
    obs: 'project', 'condition', 'method'
    uns: 'counts_from_abundance'
    layers: None (.X)

&nbsp;

&nbsp;

&nbsp;

## `pyDESeq2`: control-condition サンプルで improved vs conventional


- 対象: control 条件 4 サンプル（各プロジェクトで conventional / improved が 1 本ずつ）
- design: `~ project + method`（プロジェクト差を調整しつつ抽出法の効果を評価）
- contrast: `improved` vs `conventional`（正の log2FC = improved で高発現）

In [17]:
# https://pydeseq2.readthedocs.io/en/latest/auto_examples/plot_step_by_step.html
dds_control = DeseqDataSet(
    adata=adata_control,
    design="~ project + method",  # adjust project effect while testing extraction method
    refit_cooks=True,
    inference=DefaultInference(n_cpus=16),
    quiet=True,
    n_cpus=16
)

dds_control.deseq2()

stat_result_control = DeseqStats(
    dds_control,
    contrast=["method", "improved", "conventional"],
    quiet=True,
)

stat_result_control.summary()

results_control_impr_vs_conv = stat_result_control.results_df
results_control_impr_vs_conv.index.name = "gene_id"

display(
    results_control_impr_vs_conv.sort_values(
        by=["log2FoldChange", "padj"], ascending=(False, False)
    ).head(20)
)

/usr/local/lib/python3.13/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_id,,,,,,
ENSG00000254684.1,9.844110,6.862630,2.560797,2.679880,7.364852e-03,1.830161e-01
ENSG00000306371.1,5.171303,5.876188,2.730301,2.152212,3.138065e-02,NaN
ENSG00000265584.1,7.098970,5.015094,1.857938,2.699280,6.948976e-03,NaN
ENSG00000298929.1,8.060606,4.924216,1.782642,2.762313,5.739341e-03,NaN
ENSG00000254713.2,27.830512,4.880781,0.936019,5.214405,1.844082e-07,3.935973e-05
ENSG00000279752.1,19.500740,4.795896,1.101813,4.352732,1.344514e-05,1.398565e-03
ENSG00000222357.1,5.486300,4.635827,1.952143,2.374738,1.756142e-02,NaN
ENSG00000308369.1,3.155225,4.602739,2.904073,1.584925,1.129833e-01,NaN
ENSG00000201209.1,10.213809,4.576990,1.441532,3.175086,1.497919e-03,6.056041e-02


In [18]:
results_control_impr_vs_conv.to_csv(
    "../../out/transcript_classification/arcRNA_candidate/arcRNA_deseq2/pydeseq2_control_impr_vs_conv_results.tsv", 
    sep="\t",
    index=True
)

In [19]:
results_control_impr_vs_conv_impr = results_control_impr_vs_conv[
    (results_control_impr_vs_conv["padj"] < 0.001)
    & (results_control_impr_vs_conv["log2FoldChange"] > 1)
]

display(results_control_impr_vs_conv_impr)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_id,,,,,,
ENSG00000070831.19,1152.845174,1.060525,0.215907,4.911945,9.017749e-07,1.542706e-04
ENSG00000261326.5,1695.808022,1.231144,0.203402,6.052765,1.423804e-09,5.778971e-07
ENSG00000229994.1,74.537281,2.356470,0.447219,5.269164,1.370467e-07,3.050394e-05
ENSG00000291129.2,171.961579,1.688215,0.308467,5.472924,4.426707e-08,1.126995e-05
ENSG00000234329.1,139.935830,1.710920,0.334893,5.108857,3.241143e-07,6.329402e-05
...,...,...,...,...,...,...
ENSG00000286327.2,184.301123,1.640784,0.321497,5.103577,3.332929e-07,6.447815e-05
ENSG00000241743.5,4517.177328,2.465606,0.199628,12.351001,4.810881e-35,4.979262e-31
ENSG00000235244.7,2538.813363,1.806997,0.211966,8.524937,1.528930e-17,2.434527e-14


In [20]:
# NEAT1
ENSG00000245532_control = results_control_impr_vs_conv.query(
    "gene_id == 'ENSG00000245532.13'"
)

display(ENSG00000245532_control)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_id,,,,,,
ENSG00000245532.13,43854.356476,4.306908,0.183337,23.491727,4.955578e-122,1.025805e-117


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. 遺伝子リストの調査

- メタ解析で取得した 遺伝子 (transcript) のリストと照らし合わせて、arcRNA の候補を調査する

In [21]:
# polars で再度読み込み
results_heat_impr_vs_conv_pl = pl.from_pandas(results_heat_impr_vs_conv.reset_index())
results_control_impr_vs_conv_pl = pl.from_pandas(results_control_impr_vs_conv.reset_index())

display(results_heat_impr_vs_conv_pl)
display(results_control_impr_vs_conv_pl)

gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
str,f64,f64,f64,f64,f64,f64
"""ENSG00000310526.1""",11.424026,0.568249,0.919404,0.618063,0.536534,0.665808
"""ENSG00000310527.1""",31.737993,-0.657797,0.562344,-1.169742,0.242105,0.373697
"""ENSG00000292994.2""",9.855178,-0.526424,0.989063,-0.532246,0.594556,null
"""ENSG00000225880.7""",172.866431,0.251399,0.258933,0.970903,0.331597,0.472554
"""ENSG00000225630.1""",112.392259,-0.898566,0.323436,-2.77819,0.005466,0.018164
…,…,…,…,…,…,…
"""ENSG00000198786.2""",8247.273636,-0.494305,0.109121,-4.529875,0.000006,0.000076
"""ENSG00000198695.2""",577.739668,-0.433585,0.683646,-0.634224,0.525935,0.656607
"""ENSG00000198727.2""",12262.023272,-0.638984,0.112103,-5.699961,1.1984e-8,5.6801e-7


gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
str,f64,f64,f64,f64,f64,f64
"""ENSG00000310526.1""",36.068945,0.43949,0.559477,0.785537,0.432139,0.829349
"""ENSG00000310527.1""",76.270628,0.310076,0.421535,0.735587,0.461982,0.845152
"""ENSG00000292994.2""",12.22807,-0.84146,0.944662,-0.890753,0.373062,0.798633
"""ENSG00000225880.7""",165.940649,0.517575,0.316232,1.63669,0.101695,0.540096
"""ENSG00000225630.1""",96.760009,-1.144115,0.374569,-3.054486,0.002254,0.082597
…,…,…,…,…,…,…
"""ENSG00000198786.2""",7237.906281,-0.077254,0.6804,-0.113541,0.909601,0.984355
"""ENSG00000198695.2""",916.945835,-0.058665,0.305362,-0.192116,0.847651,0.971796
"""ENSG00000198727.2""",9258.99927,-0.146723,0.270253,-0.542909,0.587192,0.895761


&nbsp;

&nbsp;

&nbsp;

## Join Transcript All list

- HN-score を算出した transcript との統合

In [22]:
HN2_all = pl.read_parquet(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/annotation/HNscore_gencode_human_261001_HN2_transcript_all_gencode_annotation.parquet"
)

pl.Config.set_tbl_rows(2)
display(HN2_all)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000560179.2""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]","""-""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1


In [23]:
HN2_all_arcRNA = HN2_all.join(
    results_heat_impr_vs_conv_pl,
    on="gene_id",
    how="inner",
    validate="m:1",
    coalesce=True
).filter(
    (pl.col("log2FoldChange") > 1) &
    (pl.col("padj") < 0.001)
).sort(
    "HN2",
    descending=True
)

print(HN2_all_arcRNA.select("transcript_id").n_unique())

pl.Config.set_tbl_rows(2)
display(HN2_all_arcRNA.head(60))

8924


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,f64,f64,f64,f64,f64,f64
"""ENST00000339922.5""",293,17,96,276,185,7,59,178,"""VHLL""","""HGNC:30666""","""protein_coding""","""ENSG00000189030.10""","[""basic"", ""Ensembl_canonical"", … ""appris_principal_1""]","""-""",5,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select"", ""confirm experimentally""]",2,533.867811,1.347358,0.197829,6.810737,9.7100e-12,1.0808e-9
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000579982.1""",207,39,160,168,136,17,98,119,"""ABHD3""","""HGNC:18718""","""retained_intron""","""ENSG00000158201.11""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,3339.132496,1.154163,0.143623,8.036084,9.2755e-16,1.8376e-13


&nbsp;

&nbsp;

&nbsp;

## Join Transcript - CRE list

In [26]:
up5p_fantabio_cre_arcRNA = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
).join(
    results_heat_impr_vs_conv_pl,
    on=["gene_id"],
    how="inner",
    validate="m:1",
    coalesce=True
).filter(
    (pl.col("padj") < 0.001) &
    (pl.col("log2FoldChange") > 1)
)

up5p_fantabio_cre_arcRNA.write_parquet(
    "../../out/transcript_classification/arcRNA_candidate/parquet_file/up5p_fantabio_cre_filtered_arcRNA_v261010.parquet"
)

print(up5p_fantabio_cre_arcRNA.select("transcript_id").n_unique())

pl.Config.set_tbl_rows(2)
display(up5p_fantabio_cre_arcRNA)

585


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,f64,f64,f64,f64,f64,f64
"""ENST00000688485.3""",280,25,101,255,183,4,64,179,null,null,"""lncRNA""","""ENSG00000289152.3""","[""TAGENE""]","""-""",1,"""URS0002A1B76A""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]",2,"""FCHS_556375""","""chr5""",57172948,"""57173431""","""FCHS_556375""","[""ENST00000823983.1""]","""-172""",-174,"[{""EH38E2373943"",""pELS""}, {""EH38E3637789"",""PLS""}]","[{""NRF1"",4150,[{""SRX2424029"",4150}, {""SRX2424732"",3860}, … {""SRX14617581"",1014}]}, {""CTCF"",2760,[{""SRX10975999"",2760}, {""SRX10976001"",2490}, … {""SRX12008350"",760}]}, … {""BCL6"",515,[{""SRX689470"",515}]}]","[{""HS_00062715"",""Psoas Muscle"",""Muscle"",""Muscle"",""SVM""}, {""HS_00062716"",""Breast variant Human Mammary Epithelial Cells (vHMEC)"",""Breast"",""Breast"",""SVM""}, … {""HS_00062725"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}]",null,null,0.885714,"""ELA""","""255,202,74""","""yellow""",229.002872,1.382862,0.2456,5.630544,1.7964e-8,7.8225e-7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000732533.1""",86,42,278,44,48,26,177,22,"""DLEU2""","""HGNC:13748""","""lncRNA""","""ENSG00000231607.16""","[""TAGENE""]","""-""",1,"""URS0002A6AC04""","""ENSEMBL_GENCODE""",null,"""ensembl_havana_lincrna_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_113149""","""chr13""",50081410,"""50081818""","""cp16@DLEU1,cp7@DLEU2""","[""ENST00000732472.1"", ""ENST00000732498.1"", … ""ENST00000469754.4""]","""0""",251,"[{""EH38E3066439"",""pELS""}]","[{""CTCF"",3099,[{""SRX10975999"",3099}, {""SRX10976007"",2580}, … {""SRX10976003"",511}]}, {""LIN54"",2391,[{""SRX10478403"",2391}, {""SRX10478402"",2237}]}, … {""E2F6"",505,[{""SRX100514"",505}]}]","[{""HS_00186179"",""HepG2"",""Liver"",""Liver"",""SVM""}, {""HS_00186180"",""Gastric"",""Stomach"",""Stomach"",""SVM""}, … {""HS_00186186"",""Breast variant Human Mammary Epithelial Cells (vHMEC)"",""Breast"",""Breast"",""SVM""}]",null,null,-0.920635,"""PLA""","""0,0,204""","""blue""",1842.76464,1.02299,0.127319,8.034874,9.3675e-16,1.8376e-13


In [25]:
down5p_fantabio_cre_arcRNA = pl.read_parquet(
    "../../out/fantabio/parquet_file/bottom5p_transcript_fantabio_v261001.parquet"
).join(
    results_control_impr_vs_conv_pl,
    on=["gene_id"],
    how="inner",
    validate="m:1",
    coalesce=True
).filter(
    (pl.col("padj") < 0.001) &
    (pl.col("log2FoldChange") > 1)
).sort(
    "HN2",
    descending=False
)

down5p_fantabio_cre_arcRNA.write_parquet(
    "../../out/transcript_classification/arcRNA_candidate/parquet_file/down5p_fantabio_cre_filtered_arcRNA_v261010.parquet"
)

print(down5p_fantabio_cre_arcRNA.select("transcript_id").n_unique())

display(down5p_fantabio_cre_arcRNA)

66


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,f64,f64,f64,f64,f64,f64
"""ENST00000496752.1""",30,183,193,-153,24,115,112,-91,"""WDR27""","""HGNC:21248""","""retained_intron""","""ENSG00000184465.18""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_373123""","""chr6""",169669923,"""169670209""","""cp3@WDR27""","[""ENST00000496752.1""]","""-484""",-484,"[{""EH38E2526461"",""dELS""}]","[{""RELA"",2704,[{""SRX2207673"",2704}, {""SRX2207672"",2555}, … {""SRX7054621"",560}]}, {""SPI1"",2611,[{""SRX4001819"",2611}, {""SRX12684447"",2307}, … {""SRX5574460"",517}]}, … {""CHD7"",508,[{""SRX3581511"",508}]}]","[{""HS_03192414"",""Fetal Muscle Trunk"",""Muscle"",""Muscle"",""DeepSilencer""}]",null,null,-0.846154,"""ELA""","""255,202,74""","""yellow""",2015.848801,1.014499,0.206446,4.914108,8.9188e-7,0.000154
"""ENST00000682609.1""",18,155,233,-137,5,110,136,-105,"""WWOX""","""HGNC:12799""","""retained_intron""","""ENSG00000186153.19""","[""RNA_Seq_supported_only""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported only""]",1,"""FCHS_160389""","""chr16""",78099200,"""78099899""","""cp3@WWOX""","[""ENST00000627394.3"", ""ENST00000684632.1"", … ""BT007445""]","""0""",0,"[{""EH38E3192061"",""PLS""}]","[{""NRF1"",4440,[{""SRX2424732"",4440}, {""SRX2424733"",4414}, … {""SRX150389"",658}]}, {""RFX1"",4370,[{""SRX2423398"",4370}, {""SRX2423399"",4290}, … {""SRX5457513"",2155}]}, … {""RXRA"",500,[{""SRX10475074"",500}]}]","[{""HS_00220022"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}, {""HS_00220023"",""HepG2"",""Liver"",""Liver"",""SVM""}, {""HS_00220024"",""Fetal Brain Female"",""Brain"",""Brain"",""SVM""}]","[{""hg19::chr16:78133293..78133341,+;hg_155083.1"",""p2@WWOX""}, {""hg19::chr16:78133536..78133629,+;hg_155084.1"",""p1@WWOX""}]",null,0.998019,"""PLA""","""204,0,0""","""red""",430.371205,1.135701,0.239908,4.7339,0.000002,0.000321
"""ENST00000705689.1""",48,173,185,-125,28,107,116,-79,"""TRIM66""","""HGNC:29005""","""protein_coding""","""ENSG00000166436.18""","[""dotter_confirmed"", ""RNA_Seq_supported_only"", … ""CCDS""]","""-""",4,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""","[""dotter confirmed"", ""RNA-Seq supported only""]",2,"""FCHS_65294""","""chr11""",8681853,"""8682445""","""cp7@TRIM66""","[""ENST00000705689.1""]","""-187""",-187,"[{""EH38E1518565"",""pELS""}, {""EH38E2943226"",""pELS""}, … {""EH38E2943228"",""pELS""}]","[{""BRD4"",1676,[{""SRX3591437"",1676}, {""SRX3591434"",1610}, {""SRX17989514"",1010}]}, {""NPM1"",1642,[{""SRX6078416"",1642}]}, … {""RAG1"",537,[{""SRX1898037"",537}]}]","[{""HS_00149561"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}, {""HS_00149562"",""Gastric"",""Stomach"",""Stomach"",""SVM""}, … {""HS_00149571"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}]",null,null,0.915789,"""ELA""","""255,202,74""","""yellow""",2121.154565,1.1258,0.223397,5.039453,4.6686e-7,0.000083
"""ENST00000705689.1""",48,173,185,-125,28,107,116,-79,"""TRIM66""","""HGNC:29005""","""protein_coding""","""ENSG00000166436.18""","[""dotter_confirmed"", ""RN